In [39]:
#!pip install nbconvert

In [1]:
# libraries
import pandas as pd
import numpy as np
import functools as ft
import seaborn as sns
import matplotlib.pyplot as plt
# modelacion
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
import lightgbm as lgb
# encoding
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler

# seleccion de modelos
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)
from sklearn.feature_selection import (
    SelectKBest, f_classif, chi2, mutual_info_classif,
    RFE, SelectFromModel, VarianceThreshold
)

#otros
import time
import warnings
import nbconvert
warnings.filterwarnings('ignore')

## carga de datos

In [2]:
dt_merge = pd.read_csv("/home/estebanm/Escritorio/docs_tfm/Datos_desde_audios_varias_fuentes/merge_variables/audio_features.csv")
dt_merge.head()


,name,duracion_s,rms_mean,rms_std,sujeto,dementia,dementia type,gender,ethnicity,datasplit,...,syntax_depth,std_words_per_sent,mean_word_length,std_word_length,function_ratio,transcript,duration,pause_prob,language,wpm
0,Abe Burrows,71.019688,0.071390,0.050389,abeburrows,1,Alzheimer,male,Caucasian/White,train,...,7.666667,75.330383,3.789030,2.132184,0.611814,to me Bob was he was worried when I became a p...,72.76,0.263634,en,195.437053
1,Aileen Hernandez,47.000000,0.144582,0.090372,aileenhernandez,1,Dementia,female,Black/African American,train,...,5.500000,13.379088,3.810811,1.850315,0.628378,This is not going to sound like very ladylike....,47.04,0.122493,en,188.775510
2,Aileen Hernandez,38.000000,0.123960,0.079574,aileenhernandez,1,Dementia,female,Black/African American,train,...,5.428571,12.290697,3.625000,1.952562,0.585938,I arrive at my first political science class. ...,38.40,0.000328,en,200.000000
3,Aileen Hernandez,90.000000,0.061281,0.049997,aileenhernandez,1,Dementia,female,Black/African American,train,...,5.357143,9.598044,4.003597,2.092292,0.575540,We are more conscious in this state now about ...,89.00,0.002673,en,187.415730
4,Alan Ramsey,55.000000,0.060310,0.051383,alanramsey,1,Dementia,male,Caucasian/White,train,...,4.500000,8.890444,4.286585,2.216252,0.640244,"I'm pretty jaundiced one, Virginia, I have to ...",55.00,0.149834,en,178.909091


In [3]:
# sacar variables que no son parte del analisis
dt_merge.drop(['language', 'transcript', 'audio_file', 'datasplit', 'sujeto', 'name'], axis=1, inplace=True)


In [4]:
df = dt_merge.copy()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353 entries, 0 to 352
Data columns (total 74 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   duracion_s                                  353 non-null    float64
 1   rms_mean                                    353 non-null    float64
 2   rms_std                                     353 non-null    float64
 3   dementia                                    353 non-null    int64  
 4   dementia type                               218 non-null    object 
 5   gender                                      216 non-null    object 
 6   ethnicity                                   216 non-null    object 
 7   f0_mean                                     353 non-null    float64
 8   f0_std                                      353 non-null    float64
 9   f0_range                                    353 non-null    float64
 10  articulation_r

# Generacion de datos

## Clustering

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer

In [7]:
class MedicalDataAugmentor:
    def __init__(self, n_neighbors=3, random_state=42):
        self.n_neighbors = n_neighbors
        self.random_state = random_state
        np.random.seed(random_state)
        self.scaler = StandardScaler()
        self.imputer = SimpleImputer(strategy='median')

    def augment_mixed_data(self, df, target_size=2000):
        """
        Aumento de datos para mixed data -- continuas + categoricas
        """

        # 1. separar variable
        target_col = 'dementia'
        cat_cols = ['dementia type', 'gender', 'ethnicity']

        # 2. codificar categoricas
        df_processed = df.copy()

        # 2.1 manejar los nat en categoricas
        for col in cat_cols:
            df_processed[col] = df_processed[col].fillna('Unknown')

        # 2.2 manejar nan en continuas
        numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
        # excluir la columna objetivo si esta presente
        numeric_cols = [col for col in numeric_cols if col != target_col]

        # imputar valores faltantes en columnas numericas
        df_processed[numeric_cols] = self.imputer.fit_transform(df_processed[numeric_cols])

        # 3. codificar categoricas
        df_encoded = df_processed.copy()
        
        label_encoders = {}

        for col in cat_cols:
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_processed[col])
            label_encoders[col] = le

        # 4. normalizar todas las variables
        scaler = StandardScaler()
        cols_to_scale = [c for c in df_encoded.columns if c not in [target_col]]
        df_scaled = scaler.fit_transform(df_encoded[cols_to_scale])

        # 5. encontrar vecinos mas cercanos
        nn = NearestNeighbors(n_neighbors=self.n_neighbors)
        nn.fit(df_scaled)

        # 6. Generar nuevas muestras
        synthetic_samples = []

        while len(synthetic_samples) < target_size:
            # seleccionar punto aleatorio
            idx = np.random.randint(0, len(df_scaled))
            point = df_scaled[idx].reshape(1, -1)

            # encontrar vecinos
            distances, indices = nn.kneighbors(point)

            # seleccionar vecio aleatorio
            neighbor_idx = np.random.choice(indices[0][1:])

            # interpolar
            alpha = np.random.random()
            new_point = point + alpha * (df_scaled[neighbor_idx] - point[0])

            # anadir ruido gaussiano pequeno
            noise = np.random.normal(0, 0.01, size=new_point.shape)
            new_point += noise

            synthetic_samples.append(new_point[0])

        # 7. revertir transformaciones
        synthetic_array = np.array(synthetic_samples[:target_size])
        synthetic_scaled = scaler.inverse_transform(synthetic_array)

        # 8. crear dataframe
        synthetic_df = pd.DataFrame(synthetic_scaled, columns= cols_to_scale)

        # 9. proces variables categoricas

        for col in cat_cols:
            #redondear valores codificados
            synthetic_df[col] = synthetic_df[col].round().astype(int)
            # clip a rangos validos
            synthetic_df[col] = synthetic_df[col].clip(0, len(label_encoders[col].classes_)-1)
            # decodificador
            synthetic_df[col] = label_encoders[col].inverse_transform(synthetic_df[col].astype(int))

        # 10. agregar variable objetivo -- basado en vecinos
        synthetic_df[target_col] = self._assign_target_values(df, synthetic_df, cat_cols)

        return synthetic_df


    def _assign_target_values(self, original_df, synthetic_df, cat_cols):
        """ 
        Asignar valores de target basados en similitud
        """
        target_values = []

        for i, synth_row in synthetic_df.iterrows():
             # calcular similitud con puntos originales
            similarities = []

            for i, orig_row in original_df.iterrows():
                sim = 0
                # similitud en variables categoricas
                for cat_col in cat_cols:
                    if synth_row[cat_col] == orig_row[cat_col]:
                        sim += 1

                # similitud en variables continuas -- simplificado
                cont_cols = [c for c in synthetic_df.columns if c not in cat_cols + ['dementia']]

                for cont_col in cont_cols[:5]:
                    sim += 1 / (1 + abs(synth_row[cont_col] - orig_row[cont_col]))
                        
                similarities.append((sim, orig_row['dementia']))
                    
            # tomar el target del mas similar
            similarities.sort(reverse=True)
            target_values.append(similarities[0][1])

        return target_values

In [8]:
# aplicacion
augmentor = MedicalDataAugmentor(n_neighbors=3)
synthetic_data = augmentor.augment_mixed_data(df, target_size=5000)
synthetic_data.head()

,duracion_s,rms_mean,rms_std,dementia type,gender,ethnicity,f0_mean,f0_std,f0_range,articulation_rate,...,noun_pron_ratio,syntax_depth,std_words_per_sent,mean_word_length,std_word_length,function_ratio,duration,pause_prob,wpm,dementia
0,89.988196,0.036822,0.041091,Alzheimer,male,Caucasian/White,114.168930,16.642410,119.825270,1.198481,...,-1.332362e+09,3.635765,7.754072,4.067011,2.223519,0.607368,89.629681,0.013319,118.579132,1
1,29.925191,0.137287,0.088627,Dementia,female,Caucasian/White,253.413488,93.671352,456.639384,1.671122,...,-2.277397e+10,3.379363,15.586891,3.685589,2.023116,0.545056,28.267641,0.499583,158.165276,1
2,29.983313,0.135232,0.086324,Dementia,female,Caucasian/White,253.728516,88.644382,426.515383,1.653689,...,-5.833245e+09,3.511172,14.075525,3.749044,2.101150,0.542580,28.926076,0.484848,159.101324,1
3,55.943104,0.138229,0.075856,Unknown,Unknown,Unknown,184.745227,52.625316,381.154553,1.998105,...,1.088529e+10,3.195723,11.222020,4.054252,2.038020,0.627019,56.658212,0.742196,165.269704,1
4,89.971841,0.036050,0.040708,Alzheimer,male,Caucasian/White,113.763308,16.533740,121.187389,1.194309,...,-2.043078e+10,3.621857,7.642944,4.061336,2.221780,0.605613,90.068489,0.009043,119.054960,1


In [9]:
synthetic_data.shape

(5000, 74)

In [10]:
synthetic_data.dementia.value_counts()

dementia
0    2589
1    2411
Name: count, dtype: int64

In [34]:
# validaciones
def validate_augmented_data(original, synthetic):
    """Validaciones básicas para datos médicos """

    print("validación de datos médicos")

    #1. estadísticas descriptivas básicas
    print("Estadisticas por variable")
    for col in original.select_dtypes(include=[np.number]).columns[:10]:
        orig_mean = original[col].mean()
        synth_mean = synthetic[col].mean()
        diff_pct = abs(orig_mean - synth_mean)/ orig_mean * 100
        print(f"{col}: Original={orig_mean:.2f}, sintetico={synth_mean:.2f}, diff={diff_pct:.1f}%")

    # 2. DIstribuciones de categóricas
    print("\n2. Distribuciones categoricas")
    for col in ['dementia type', 'gender', 'ethnicity', 'dementia']:
        print(f"\n{col}")
        orig_dist = original[col].value_counts(normalize=True)
        synth_dist = synthetic[col].value_counts(normalize=True)

        for val in orig_dist.index:
            orig_pct = orig_dist[val] * 100
            synth_pct = synth_dist.get(val, 0) * 100
            print(f" {val}: original={orig_pct:.1f}%, sintetico={synth_pct:.1f}%")

    # 3. Correlaciones clave

    print("\n3. Correlaciones principales:")
    key_cols = ['f0_mean', 'articulation_rate', 'HNRdBACF_sma3nz_amean', 'dementia']
    
    orig_corr = original[key_cols].corr()
    synth_corr = synthetic[key_cols].corr()
    
    print("Correlación dementia-f0_mean:")
    print(f"  Original: {orig_corr.loc['dementia', 'f0_mean']:.3f}")
    print(f"  Sintético: {synth_corr.loc['dementia', 'f0_mean']:.3f}")
    
    return True

In [35]:
validate_augmented_data(df, synthetic_data)


validación de datos médicos
Estadisticas por variable
duracion_s: Original=56.02, sintetico=56.06, diff=0.1%
rms_mean: Original=0.09, sintetico=0.09, diff=1.5%
rms_std: Original=0.06, sintetico=0.06, diff=0.3%
dementia: Original=0.37, sintetico=0.48, diff=29.9%
f0_mean: Original=178.47, sintetico=175.37, diff=1.7%
f0_std: Original=48.84, sintetico=46.96, diff=3.8%
f0_range: Original=300.61, sintetico=290.51, diff=3.4%
articulation_rate: Original=1.85, sintetico=1.85, diff=0.1%
silence_segments_n: Original=54.00, sintetico=53.91, diff=0.2%
silence_dur_mean_s: Original=0.28, sintetico=0.28, diff=1.6%

2. Distribuciones categoricas

dementia type
 no dementia: original=40.8%, sintetico=18.8%
 Alzheimer: original=27.5%, sintetico=11.2%
 Dementia: original=24.3%, sintetico=12.8%
 Lewy body: original=5.0%, sintetico=6.4%
 Parkinson: original=1.4%, sintetico=5.0%
 Primary progressive aphasia: original=0.9%, sintetico=5.2%

gender
 male: original=85.6%, sintetico=47.7%
 female: original=14.4%,

True

In [36]:

def validate_augmented_data(original, synthetic):
    """
    Validate synthetic medical data against original data.
    
    Parameters:
    - original: pd.DataFrame - original dataset
    - synthetic: pd.DataFrame - synthetic dataset
    
    Returns:
    - pd.DataFrame with columns:
        type: type of metric ('numeric_mean', 'categorical', 'correlation')
        variable: column name or correlation pair
        category: category level (for categorical) or NaN
        original: original value (mean, proportion, correlation)
        synthetic: synthetic value
        diff_pct: percent difference (only for numeric means, else NaN)
    """
    rows = []
    
    # 1. Numeric descriptive statistics (first 10 numeric columns)
    numeric_cols = original.select_dtypes(include=[np.number]).columns[:10]
    for col in numeric_cols:
        orig_mean = original[col].mean()
        synth_mean = synthetic[col].mean()
        # Avoid division by zero if original mean is 0
        if orig_mean != 0:
            diff_pct = abs(orig_mean - synth_mean) / orig_mean * 100
        else:
            diff_pct = np.nan
        rows.append({
            'type': 'numeric_mean',
            'variable': col,
            'category': np.nan,
            'original': orig_mean,
            'synthetic': synth_mean,
            'diff_pct': diff_pct
        })
    
    # 2. Categorical distributions
    cat_cols = ['dementia type', 'gender', 'ethnicity', 'dementia']
    for col in cat_cols:
        if col not in original.columns or col not in synthetic.columns:
            continue
        orig_dist = original[col].value_counts(normalize=True)
        synth_dist = synthetic[col].value_counts(normalize=True)
        # Union of all categories present in either dataset
        all_cats = set(orig_dist.index) | set(synth_dist.index)
        for cat in all_cats:
            orig_pct = orig_dist.get(cat, 0) * 100
            synth_pct = synth_dist.get(cat, 0) * 100
            rows.append({
                'type': 'categorical',
                'variable': col,
                'category': cat,
                'original': orig_pct,
                'synthetic': synth_pct,
                'diff_pct': abs(orig_pct - synth_pct)   # absolute percentage point difference
            })
    
    # 3. Key correlations
    key_cols = ['f0_mean', 'articulation_rate', 'HNRdBACF_sma3nz_amean', 'dementia']
    # Ensure all columns exist
    key_cols = [c for c in key_cols if c in original.columns and c in synthetic.columns]
    if len(key_cols) >= 2:
        orig_corr = original[key_cols].corr()
        synth_corr = synthetic[key_cols].corr()
        # We'll report the correlation between dementia and each other key column
        if 'dementia' in key_cols:
            for col in key_cols:
                if col == 'dementia':
                    continue
                orig_val = orig_corr.loc['dementia', col]
                synth_val = synth_corr.loc['dementia', col]
                rows.append({
                    'type': 'correlation',
                    'variable': f'dementia_{col}',
                    'category': np.nan,
                    'original': orig_val,
                    'synthetic': synth_val,
                    'diff_pct': abs(orig_val - synth_val) * 100   # absolute difference in correlation (as percentage points)
                })
    
    # Create DataFrame
    result_df = pd.DataFrame(rows)
    
    # Optional: keep the original print statements for immediate feedback
    print("Validación de datos médicos")
    print("\nEstadísticas por variable:")
    for col in numeric_cols:
        row = result_df[(result_df['type']=='numeric_mean') & (result_df['variable']==col)].iloc[0]
        print(f"{col}: Original={row['original']:.2f}, Sintético={row['synthetic']:.2f}, diff={row['diff_pct']:.1f}%")
    
    print("\nDistribuciones categóricas:")
    for col in cat_cols:
        print(f"\n{col}")
        cat_rows = result_df[(result_df['type']=='categorical') & (result_df['variable']==col)]
        for _, row in cat_rows.iterrows():
            print(f" {row['category']}: original={row['original']:.1f}%, sintético={row['synthetic']:.1f}%")
    
    print("\nCorrelaciones principales:")
    corr_rows = result_df[result_df['type']=='correlation']
    for _, row in corr_rows.iterrows():
        print(f"{row['variable']}: Original={row['original']:.3f}, Sintético={row['synthetic']:.3f}, diff={row['diff_pct']:.2f}%")
    
    return result_df

In [37]:
dcluster = validate_augmented_data(df, synthetic_data)
dcluster

Validación de datos médicos

Estadísticas por variable:
duracion_s: Original=56.02, Sintético=56.06, diff=0.1%
rms_mean: Original=0.09, Sintético=0.09, diff=1.5%
rms_std: Original=0.06, Sintético=0.06, diff=0.3%
dementia: Original=0.37, Sintético=0.48, diff=29.9%
f0_mean: Original=178.47, Sintético=175.37, diff=1.7%
f0_std: Original=48.84, Sintético=46.96, diff=3.8%
f0_range: Original=300.61, Sintético=290.51, diff=3.4%
articulation_rate: Original=1.85, Sintético=1.85, diff=0.1%
silence_segments_n: Original=54.00, Sintético=53.91, diff=0.2%
silence_dur_mean_s: Original=0.28, Sintético=0.28, diff=1.6%

Distribuciones categóricas:

dementia type
 Lewy body: original=5.0%, sintético=6.4%
 Parkinson: original=1.4%, sintético=5.0%
 Alzheimer: original=27.5%, sintético=11.2%
 Dementia: original=24.3%, sintético=12.8%
 no dementia: original=40.8%, sintético=18.8%
 Primary progressive aphasia: original=0.9%, sintético=5.2%
 Unknown: original=0.0%, sintético=40.6%

gender
 female: original=14.4

,type,variable,category,original,synthetic,diff_pct
0,numeric_mean,duracion_s,NaN,56.019615,56.055137,0.063411
1,numeric_mean,rms_mean,NaN,0.089943,0.088552,1.546447
2,numeric_mean,rms_std,NaN,0.061699,0.061505,0.315503
3,numeric_mean,dementia,NaN,0.371105,0.482200,29.936336
4,numeric_mean,f0_mean,NaN,178.469345,175.367931,1.737786
5,numeric_mean,f0_std,NaN,48.843398,46.964070,3.847660
6,numeric_mean,f0_range,NaN,300.610748,290.512940,3.359097
7,numeric_mean,articulation_rate,NaN,1.847707,1.845993,0.092736
8,numeric_mean,silence_segments_n,NaN,53.997167,53.911997,0.157731
9,numeric_mean,silence_dur_mean_s,NaN,0.281062,0.276447,1.642125


In [13]:
data_aumentada_comp = pd.concat([df, synthetic_data ], ignore_index=True)

In [14]:
data_aumentada_comp.head()

,duracion_s,rms_mean,rms_std,dementia,dementia type,gender,ethnicity,f0_mean,f0_std,f0_range,...,propn_ratio,noun_pron_ratio,syntax_depth,std_words_per_sent,mean_word_length,std_word_length,function_ratio,duration,pause_prob,wpm
0,71.019688,0.071390,0.050389,1,Alzheimer,male,Caucasian/White,174.444418,51.103025,311.055420,...,0.037975,0.465517,7.666667,75.330383,3.789030,2.132184,0.611814,72.76,0.263634,195.437053
1,47.000000,0.144582,0.090372,1,Dementia,female,Black/African American,292.169978,115.242586,659.783615,...,0.000000,0.500000,5.500000,13.379088,3.810811,1.850315,0.628378,47.04,0.122493,188.775510
2,38.000000,0.123960,0.079574,1,Dementia,female,Black/African American,238.856939,58.060892,352.463261,...,0.000000,0.655172,5.428571,12.290697,3.625000,1.952562,0.585938,38.40,0.000328,200.000000
3,90.000000,0.061281,0.049997,1,Dementia,female,Black/African American,255.297147,68.547493,638.292396,...,0.000000,1.146341,5.357143,9.598044,4.003597,2.092292,0.575540,89.00,0.002673,187.415730
4,55.000000,0.060310,0.051383,1,Dementia,male,Caucasian/White,134.696479,38.961565,243.928633,...,0.067073,1.000000,4.500000,8.890444,4.286585,2.216252,0.640244,55.00,0.149834,178.909091


In [15]:
data_aumentada_comp.shape

(5353, 74)

In [17]:
data_aumentada_comp.to_csv('data_aumentada_v1.csv')

****

## CopulaGAN 

In [38]:
import numpy as np
from scipy import stats
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler

In [39]:
def generate_with_copula(df, n_samples=2000):
        """
        Copulas gaussianas
        """

        df_filled = df.copy()

        # imputar categoricas
        categorical_cols = ['dementia', 'dementia type', 'gender', 'ethnicity']
        for col in categorical_cols:
            if col in df_filled.columns:
                df_filled[col] = df_filled[col].fillna('Unknown')

        # imputar numericas con la mediana
        numeric_cols = df_filled.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df_filled[col] = df_filled[col].fillna(df_filled[col].median())
    
        # identificar variables continuos 
        continuous_cols = df_filled.select_dtypes(include=[np.number]).columns.tolist()
        # excluir columnas que no son realmente continuasl
        exclude_cols = ['dementia'] + categorical_cols
        continuous_cols = [c for c in continuous_cols if c not in exclude_cols]

        print(f"procesando {len(continuous_cols)} varables continuas")

        # transformar variables continuas a normales

    
        transformed_data = []
        transformers = {}

        for col in continuous_cols:
           try:
                # usar transformacion de potencia para normalizar
                pt = PowerTransformer(method='yeo-johnson')
                transformed = pt.fit_transform(df_filled[[col]]) 
                transformed_data.append(transformed.flatten())
                transformers[col] = pt
           except Exception as e:
                print(f"Error transformando {col} : {e}")
                # si falla la tranformacion usar estandarizacion simple
                scaler = StandardScaler()
                tranformed = scaler.fit_transform(df_filled[[col]])
                transformed_data.append(transformed.flatten())
                transformers[col] = scaler
           
        #  Asegurar que los arrays tenga la misma longitud
        min_length = min(len(arr) for arr in transformed_data)
        if min_length < len(df_filled):
            print(f"advertencia: algunas columnas tenian NaN. Usando {min_length} muestras validas")


        # recordar todos los arrays a la misma longitud 
        transformed_data = [arr[:min_length] for arr in transformed_data]


        # calcular la matriz de correlacion
        transformed_matrix = np.column_stack(transformed_data)
        corr_matrix = np.corrcoef(transformed_matrix, rowvar=False)

        # generar datos normales multivariados
        mean_vector = np.zeros(len(continuous_cols))
        try:
            synthetic_normal = np.random.multivariate_normal(mean_vector, 
                                                         corr_matrix, 
                                                         n_samples)

        except Exception as e:
            print(f"eror en mumtivarite_normal: {e}")
            print("Generando variables independientes como fallback")
            synthetic_normal = np.random.randn(n_samples, len(continuous_cols))



    
        # Transformar de vuelta a la distribucion original
        synthetic_df = pd.DataFrame()

        for i, col in enumerate(continuous_cols):
            pt = transformers[col]
            synthetic_col = pt.inverse_transform(synthetic_normal[:, i].reshape(-1, 1))
            synthetic_df[col] = synthetic_col.flatten()

        # agregar variables categoricas
        for cat_col in categorical_cols + ['dementia']:
            if cat_col in df_filled.columns:
                value_counts = df_filled[cat_col].value_counts(normalize=True)

                if len(value_counts) > 0:
            # samplear de distribucion empirica
       
                    synthetic_df[cat_col] = np.random.choice(
                                    value_counts.index.tolist(),
                                    size=n_samples,
                                    p=value_counts.values)
        else:
            synthetic_df[cat_col] = df_filled[cat_col].mode()[0] if len(df_filled[cat_col].mode()) > 0 else 'Unknown'

        for col in categorical_cols:
            if col in synthetic_df.columns:
                synthetic_df[col] = synthetic_df[col].astype(str)
        if 'dementia' in synthetic_df.columns:
            synthetic_df['dementia'] = synthetic_df['dementia'].astype(int)

        print(f"Datos generados: {synthetic_df.shape}")

    
        return synthetic_df

In [40]:
data_copula = generate_with_copula(df)
data_copula.head()

procesando 70 varables continuas
Datos generados: (2000, 74)


,duracion_s,rms_mean,rms_std,f0_mean,f0_std,f0_range,articulation_rate,silence_segments_n,silence_dur_mean_s,silence_ratio_rms,...,mean_word_length,std_word_length,function_ratio,duration,pause_prob,wpm,dementia,dementia type,gender,ethnicity
0,112.441933,0.085655,0.061826,132.614485,38.533583,268.792227,2.051580,114.735072,0.298082,-0.038617,...,4.125818,2.759257,0.703589,114.479147,-0.062814,129.895398,0,no dementia,male,Caucasian/White
1,10.174921,0.146490,0.089174,178.375550,82.301930,274.780598,2.338567,5.591024,0.448191,NaN,...,3.645223,1.699591,0.578712,14.301604,0.228810,132.711440,0,no dementia,Unknown,Caucasian/White
2,92.817704,0.085214,0.049355,237.280122,66.703894,530.393934,1.962066,122.590275,0.197355,-0.046654,...,3.778772,2.112460,0.536747,89.373133,0.076410,150.969087,0,Alzheimer,male,Caucasian/White
3,50.793531,0.099657,0.079062,190.158573,40.968400,216.621076,1.792133,50.502122,0.256979,-0.024007,...,4.101414,2.108838,0.630283,46.652842,0.183997,128.658333,0,Unknown,Unknown,Unknown
4,37.450149,0.364231,0.164388,352.698087,141.900151,800.511506,1.828856,23.334554,0.311136,NaN,...,3.604392,1.785155,0.582888,41.006805,0.280958,147.492738,0,Unknown,male,Caucasian/White


In [24]:
data_copula.dementia.value_counts()

dementia
0    2000
Name: count, dtype: int64

In [42]:
dcopulas = validate_augmented_data(df, data_copula)
dcopulas

Validación de datos médicos

Estadísticas por variable:
duracion_s: Original=56.02, Sintético=56.32, diff=0.5%
rms_mean: Original=0.09, Sintético=0.09, diff=0.2%
rms_std: Original=0.06, Sintético=0.06, diff=0.3%
dementia: Original=0.37, Sintético=0.00, diff=100.0%
f0_mean: Original=178.47, Sintético=179.79, diff=0.7%
f0_std: Original=48.84, Sintético=49.98, diff=2.3%
f0_range: Original=300.61, Sintético=313.27, diff=4.2%
articulation_rate: Original=1.85, Sintético=1.85, diff=0.1%
silence_segments_n: Original=54.00, Sintético=54.57, diff=1.1%
silence_dur_mean_s: Original=0.28, Sintético=0.28, diff=0.9%

Distribuciones categóricas:

dementia type
 Lewy body: original=5.0%, sintético=3.8%
 Parkinson: original=1.4%, sintético=0.7%
 Alzheimer: original=27.5%, sintético=16.2%
 Dementia: original=24.3%, sintético=15.8%
 no dementia: original=40.8%, sintético=24.3%
 Primary progressive aphasia: original=0.9%, sintético=0.6%
 Unknown: original=0.0%, sintético=38.8%

gender
 female: original=14.

,type,variable,category,original,synthetic,diff_pct
0,numeric_mean,duracion_s,NaN,56.019615,56.319882,0.536004
1,numeric_mean,rms_mean,NaN,0.089943,0.089764,0.199666
2,numeric_mean,rms_std,NaN,0.061699,0.061491,0.337350
3,numeric_mean,dementia,NaN,0.371105,0.000000,100.000000
4,numeric_mean,f0_mean,NaN,178.469345,179.789760,0.739855
5,numeric_mean,f0_std,NaN,48.843398,49.980694,2.328455
6,numeric_mean,f0_range,NaN,300.610748,313.265348,4.209630
7,numeric_mean,articulation_rate,NaN,1.847707,1.846153,0.084100
8,numeric_mean,silence_segments_n,NaN,53.997167,54.571583,1.063788
9,numeric_mean,silence_dur_mean_s,NaN,0.281062,0.278610,0.872486


****

## Mixtura Gaussiana

In [43]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
import warnings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
warnings.filterwarnings('ignore')

In [44]:
class GMMDataGenerator:
    def __init__(self, n_components=None, covariance_type='full', imputation_strategy='median',
                random_state=42):
        """ 
        Generador de datos usando gaussian misture models GMM
        """
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.imputation_strategy = imputation_strategy
        self.random_state = random_state
        self.label_encoders = {}
        self.label_encoders = {}
        self.gmm = None
        self.imputer = None
        np.random.seed(random_state)

    def _handle_missing_values(self, df):
        """ 
        Manejo avanzado de valores faltantes
        """

        df_clean = df.copy()

        # identificar columnas con NaN
        nan_columns = df_clean.columns[df_clean.isna().any()].tolist()

        if not nan_columns:
            print("No se encontraron valoes NaN en el dataset")
            return df_clean

        print(f"columnas con NaN: {len(nan_columns)}")

        # separar columnas categoricas y numericas
        categorical_cols = ['dementia type', 'gender', 'ethnicity']
        numerical_cols = [col for col in df_clean.columns
                         if col not in categorical_cols and np.issubdtype(df_clean[col].dtype, np.number)]
        
        # manejo de NaN en categoricas
        for col in categorical_cols:
            if col in df_clean.columns and col in nan_columns:
                nan_count = df_clean[col].isna().sum()
                print(f"- {col} : {nan_count} NaN ({nan_count/len(df_clean)*100:.1f}%)")

                # usar moda para imputar
                if not df_clean[col].mode().empty:
                    mode_val = df_clean[col].mode()[0]
                    df_clean[col] = df_clean[col].fillna(mode_val)
                    print(f" rellenado con '{mode_val}'")
                else:
                    df_clean[col] = df_clean[col].fillna("Unknown")
                    print(f" rellenado con unknowm")
                    
        # manejo de nan en numericas
        numerical_nan_cols = [col for col in numerical_cols if col in nan_columns]

        if numerical_nan_cols:
            print(f"\n Imputando {len(numerical_nan_cols)} columnas numericas")

            if self.imputation_strategy == 'knn':
                self.imputer = KNNImputer(n_neighbors=5)
            elif self.imputation_strategy == 'iterative':
                self.imputer = IterativeImputer(max_iter=10, random_state= self.random_state)
            else:
                self.imputer = SimpleImputer(strategy=self.imputation_strategy)

            # imputar solo columnas numericas
            df_clean[numerical_nan_cols] = self.imputer.fit_transform(df_clean[numerical_nan_cols])
            print(f"Estrategia: {self.imputation_strategy}")

        # verificar que no queden NaN
        remaining_nan = df_clean.isna().sum().sum()
        if remaining_nan > 0:
            print(f"Adventencia: {remaining_nan} valores NaN restantes, usando fillna(0)")
            df_clean = df_clean.fillna(0)

        return df_clean
            
          
    def _encode_categorical_variables(self, df):

        """
        codigicar variables categoricas preservando mapeos
        """
        df_encoded = df.copy()
        categorical_cols = ['dementia type', 'gender', 'ethnicity']

        for col in categorical_cols:
            if col in df_encoded.columns:
                unique_vals = df_encoded[col].unique()

                le = LabelEncoder()
                df_encoded[col] = le.fit_transform(df_encoded[col])
                self.label_encoders[col] = {
                    'encoder': le,
                    'clases': le.classes_,
                    'original_values':unique_vals
                }

                print(f"{col}: {len(unique_vals)} categorias codificadas")
                
        return df_encoded


    def _decode_categorical_variables(self, df_encoded):
         """ 
         Decodificar varaibles categoricas usando los mapeos guardados
         """
         df_decoded = df_encoded.copy()

         for col, encoder_info, in self.label_encoders.items():
             if col in df_encoded.columns:
                 le = encoder_info['encoder']

                 # redondear valores continuos a enteros
                 df_decoded[col] = df_decoded[col].round().astype(int)

                 n_clases = len(le.classes_)
                 df_decoded[col] = df_decoded[col].clip(0, n_clases -1)

                 # decodificar
                 try:
                     df_decoded[col] = le.inverse_transform(df_decoded[col])
                 except ValueError as e:
                     print(f"Error decodificado {col}:{e}")
                     print(f"Valores unicos en datos codificados: {df_decoded[col].unique()}")
                     print(f"Clases del encoder: {le.classes_}") 

                     # fallback mapear el valor mas cercano
                     def safe_inverse(val):
                         try:
                             return le.inverse_transform([val])[0]
                         except:
                              return le.classes_[0]

                     df_decoded[col] = df_decoded[col].apply(safe_inverse)

         return df_decoded

    def fit(self, df, target_col='dementia'):
          """
          Ajustar el modelo GMM a los datos
          """
          # manejo valores faltantes
          df_clean = self._handle_missing_values(df)

          # separar caracteristicas y target si exist
          if target_col in df_clean.columns:
              X = df_clean.drop(columns=[target_col])
              self.y =df_clean[target_col]

              print(f"variable objetivo '{target_col}' separada")
          else:
              X = df_clean
              self.y = None
              print("No se encontro variable objetivo")
              
          # codificar categoricas
          X_encoded = self._encode_categorical_variables(X)

          # determinar numero de componentes
          if self.n_components is None:
              n_samples = len(X_encoded)
              n_features = X_encoded.shape[1]

              # regla general: sqrt(n_samples) o n_samples//10
              self.n_components = max(2, min(10, int(np.sqrt(n_samples)), n_samples //10))
              print(f" numero de componentes automatico: {self.n_components}")
          else:
              print(f" numero de componentes especificado: {self.n_components}")

          # entrena GMM
          self.gmm = GaussianMixture(
              n_components = self.n_components,
              covariance_type = self.covariance_type,
              random_state = self.random_state,
              init_params='kmeans',
              max_iter = 200,
              tol=1e-3,
              verbose=1
          )
              
          self.gmm.fit(X_encoded)
          print(f"covergio:{self.gmm.converged_}")
          print(f"iteraciones:{self.gmm.n_iter_}")
          print(f"score: {self.gmm.score(X_encoded):.4f}")

          # guardar los datos transformados para referencia
          self.X_encoded = X_encoded
          self.X_original = X

          print(f"Modelo GMM ajustado exitosamente")

          return self


    def generate_samples(self, n_samples=1000, include_target=True, target_distribution='match'):
        """
        Generar muestras sintéticas
        
        Parameters:
        -----------
        n_samples : int
            Número de muestras a generar
        include_target : bool
            Si True, incluye la variable objetivo en la salida
        target_distribution : str
            'match' : mantiene distribución original de target
            'balanced' : balancea las clases
            'random' : asigna aleatoriamente según probabilidades originales
        """

        if self.gmm is None:
            raise ValueError("El modelo GMM no ha sido ajustado")

        # generar muestras del GMM
        synthetic_encoded, cluster_labels = self.gmm.sample(n_samples)
        synthetic_encoded = synthetic_encoded.astype(float)

        # crear dataframe
        synthetic_df = pd.DataFrame(synthetic_encoded, columns=self.X_encoded.columns)

        # decodificar variables categoricas
        synthetic_df = self._decode_categorical_variables(synthetic_df)

        # asignar variable objetivo si es necesario
        if include_target and self.y is not None:
            if target_distribution == 'match':
                # mantener distribucion original por cluster
                synthetic_df['dementia'] = self._assign_target_by_cluster(
                    cluster_labels, self.gmm.weights_
                )
            elif target_distribution == 'balanced':
                # asignar clases balanceadas
                n_clases =lent(np.unique(self.y))
                synthetic_df['dementia'] = np.random.randint(0, n_clases, n_samples)
            else:
                # asignar segun distribucion original
                probs = np.bincount(self.y)/ len(self.y)
                synthetic_df['dementia'] = np.random.choice(len(probs), n_samples, p=probs)
                
        return synthetic_df


    def _assign_target_by_cluster(self, cluster_labels, cluster_weights):
            """ 
            Asignar valores de target basados en distribucion por cluster
            """

            target_values = []

            # para cada cluster, calcular distribucion de target
            cluster_target_dist = {}
            for cluster in range(self.n_components):
                # encontrar muestras originales en est cluster
                cluster_probs = self.gmm.predict_proba(self.X_encoded)
                cluster_mask = cluster_probs.argmax(axis=1) == cluster

                if cluster_mask.sum() > 0:
                    # distribucion de target en este cluster
                    cluster_y = self.y[cluster_mask]
                    unique, counts = np.unique(cluster_y, return_counts=True)
                    dist = counts / counts.sum()
                    cluster_target_dist[cluster] = (unique, dist)
                else:
                    #si no hay muestras, usar distribucion global
                    unique, counts = np.unique(self.y, return_counts=True)
                    dist = counts / counts.sum()
                    cluster_target_dist[cluster] = (unique, dist)

            # asignar target a cada muestra sintetica
            for cluster in cluster_labels:
                unique, dist = cluster_target_dist[cluster]
                target_values.append(np.random.choice(unique, p=dist))

            return np.array(target_values)

    def validate_generated_data(self, synthetic_df, original_df=None, n_tests=5):
        """ 
        validar calidad de datos generados
        """

        if original_df is None:
            original_df = self.X_original

        # 1. estatisticas basicas
        print(f"\n estadisticas basicas")
        print(f" original: {len(original_df)} muestras")
        print(f" sintetico: {len(synthetic_df)} muestras")

        # 2. Distribucion de variables categoricas

        print("\n distribuciones de variables categoricas:")
        cat_cols = ['dementia type', 'gender', 'ethnicity', 'dementia']

        for col in cat_cols:
            if col in synthetic_df.columns and col in original_df.columns:
                print(f"\n {col}")
                orig_dist = original_df[col].value_counts(normalize=True).sort_index()
                synth_dist = synthetic_df[col].value_counts(normalize=True).sort_index()

                for val in orig_dist.index[:5]:
                    orig_pct = orig_dist[val] * 100
                    synth_val = synth_dist.index[synth_dist.index == val]
                    if len(synth_val) > 0:
                        synth_pct = synth_dist[synth_val[0]] * 100
                    else:
                        synth_pct = 0
                    print(f"{val}: orignal={orig_pct:.1f}%, sintetico={synth_pct:1f}%")

        # 3. media y desviacion estandar de variables numericas clave
        print(" variables numericas clave: ")

        num_cols = ['f0_mean', 'articulation_rate', 'HNRdBACF_sma3nz_amean']

        for col in num_cols:
            if col in synthetic_df.columns and col in original_df.columns:
                print(f"{col}:")
                orig_mean = original_df[col].mean()
                synth_mean = synthetic_df[col].mean()
                orig_std = original_df[col].std()
                synth_std = synthetic_df[col].std()

                print(f"Media: original={orig_mean:.4f}, sintetico={synth_mean:.4f}")
                print(f"Std: original={orig_std:.4f}, sintetico={synth_std:.4f}")


                # diferencia porcentual
                mean_diff_pct = abs(orig_mean - synth_mean) / orig_mean * 100
                std_diff_pct = abs(orig_std - synth_mean) / orig_std * 100
                print(f" Dif media: {mean_diff_pct:.1f}%")
                print(f" Dif std: {std_diff_pct:.1f}%")
        return True                   

In [45]:
def main():

    # crear instancia del generador
    generator = GMMDataGenerator(
        n_components=5, 
        covariance_type='full',
        imputation_strategy='median',
        random_state = 42
    )

    # ajuste del modelo
    generator.fit(df, target_col='dementia')

    # generar datos sinteticos
    print(f"Generando datos sinteticos")
    synthetic_data = generator.generate_samples(n_samples=2000, 
                                                include_target=True, 
                                                target_distribution='match')
    # validar datos generados
    generator.validate_generated_data(synthetic_data)

    # mostrar primeros registros
    print("\nPrimeras 10 filas de datos sinteticos:")
    print(synthetic_data.head(10))

    # gurdar datos si es necesario
    synthetic_data.to_csv('datos_synteticos_gmm.csv', index=False)
    print("\nDatos guardatos en 'datos_synteticos_gmm.csv'")

    return synthetic_data

# ejecucion directa
if __name__ == "__main__":
    synthetic_data = main()

columnas con NaN: 19
- dementia type : 135 NaN (38.2%)
 rellenado con 'no dementia'
- gender : 137 NaN (38.8%)
 rellenado con 'male'
- ethnicity : 137 NaN (38.8%)
 rellenado con 'Caucasian/White'

 Imputando 16 columnas numericas
Estrategia: median
variable objetivo 'dementia' separada
dementia type: 6 categorias codificadas
gender: 2 categorias codificadas
ethnicity: 5 categorias codificadas
 numero de componentes especificado: 5
Initialization 0
Initialization converged.
covergio:True
iteraciones:2
score: 6.3783
Modelo GMM ajustado exitosamente
Generando datos sinteticos

 estadisticas basicas
 original: 353 muestras
 sintetico: 2000 muestras

 distribuciones de variables categoricas:

 dementia type
Alzheimer: orignal=17.0%, sintetico=7.350000%
Dementia: orignal=15.0%, sintetico=9.150000%
Lewy body: orignal=3.1%, sintetico=14.100000%
Parkinson: orignal=0.8%, sintetico=19.200000%
Primary progressive aphasia: orignal=0.6%, sintetico=18.050000%

 gender
female: orignal=8.8%, sintetico=

In [46]:
dmixtura = validate_augmented_data(df, synthetic_data)
dmixtura

Validación de datos médicos

Estadísticas por variable:
duracion_s: Original=56.02, Sintético=56.40, diff=0.7%
rms_mean: Original=0.09, Sintético=0.09, diff=1.7%
rms_std: Original=0.06, Sintético=0.06, diff=1.6%
dementia: Original=0.37, Sintético=0.37, diff=0.5%
f0_mean: Original=178.47, Sintético=182.09, diff=2.0%
f0_std: Original=48.84, Sintético=48.75, diff=0.2%
f0_range: Original=300.61, Sintético=301.75, diff=0.4%
articulation_rate: Original=1.85, Sintético=1.85, diff=0.3%
silence_segments_n: Original=54.00, Sintético=54.68, diff=1.3%
silence_dur_mean_s: Original=0.28, Sintético=0.28, diff=0.5%

Distribuciones categóricas:

dementia type
 Lewy body: original=5.0%, sintético=14.1%
 Parkinson: original=1.4%, sintético=19.2%
 Alzheimer: original=27.5%, sintético=7.3%
 Dementia: original=24.3%, sintético=9.2%
 no dementia: original=40.8%, sintético=32.1%
 Primary progressive aphasia: original=0.9%, sintético=18.1%

gender
 male: original=85.6%, sintético=93.3%
 female: original=14.4%,

,type,variable,category,original,synthetic,diff_pct
0,numeric_mean,duracion_s,NaN,56.019615,56.395245,0.670534
1,numeric_mean,rms_mean,NaN,0.089943,0.091462,1.688736
2,numeric_mean,rms_std,NaN,0.061699,0.062682,1.592101
3,numeric_mean,dementia,NaN,0.371105,0.373000,0.510687
4,numeric_mean,f0_mean,NaN,178.469345,182.093486,2.030680
5,numeric_mean,f0_std,NaN,48.843398,48.747999,0.195316
6,numeric_mean,f0_range,NaN,300.610748,301.751875,0.379603
7,numeric_mean,articulation_rate,NaN,1.847707,1.853671,0.322770
8,numeric_mean,silence_segments_n,NaN,53.997167,54.683214,1.270523
9,numeric_mean,silence_dur_mean_s,NaN,0.281062,0.279742,0.469833


****

## Método Bayesiano para Data Augmentation

In [48]:
import pandas as pd
import numpy as np
from scipy.stats import norm, multivariate_normal
from statsmodels.distributions.copula.api import GaussianCopula
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [49]:
# Preprocesamiento similar al anterior
num_cols = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] and col != 'dementia']
cat_cols = ['dementia type', 'gender', 'ethnicity']
target_col = 'dementia'

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# separar por clase para modelar condicionalmente
df_0 = df[df[target_col] == 0].drop(columns=[target_col])
df_1 = df[df[target_col] == 1].drop(columns=[target_col])

X_0 = preprocessor.fit_transform(df_0) 
X_1 = preprocessor.transform(df_1)

# funcion para fit copula por clase

def fit_gaussian_copula(X, epsilon=1e-6):
    if X.shape[0] < 2:
        raise ValueError("need at leat 2 rows to fit a copula")
    # transformar a uniformes
    u = norm.cdf(X)
    u = np.clip(u, epsilon, 1 - epsilon)
    u += np.random.uniform(-1e-8, 1e-8, size=u.shape)
    u = np.clip(u, epsilon, 1 - epsilon)
    # correlacion para gaussian copula
    corr = np.corrcoef(u.T)
    # arreglar NaN
    if not np.all(np.isfinite(corr)):
        print("warning:correlation matrix contained inf/nan")
        # replace nana with 0, inf with +/- 1
        corr = np.nan_to_num(corr, nan=0.0, posinf=1.0, neginf=-1.0)
        corr = np.clip(corr, -1.0, 1.0)
    
    corr = (corr + corr.T) / 2
    # ridge regularization
    corr += 1e-5 * np.eye(corr.shape[0])
    
    # hacerla semidefinida positiva
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.maximum(eigvals, 1e-8)
    corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    # reconstruir matrix
    corr_psd = (corr_psd + corr_psd.T) / 2
    corr_psd = np.clip(corr_psd, -1.0, 1.0)
    copula = GaussianCopula(corr=corr_psd, allow_singular=True)
    return copula, u

copula_0, u_0 = fit_gaussian_copula(X_0)
copula_1, u_1 = fit_gaussian_copula(X_1)

# Generar muestras nuevas
def generate_copula_samples(copula, num_samples, preprocessor):
    synth_u = copula.rvs(nobs=num_samples)
    synth_x = norm.ppf(synth_u)

    num_pipe = preprocessor.named_transformers_['num']
    cat_pipe = preprocessor.named_transformers_['cat']

    n_num = len(num_pipe.named_steps['scaler'].mean_)

    synth_num_scaled = synth_x[:, :n_num]
    synth_cat_encoded = synth_x[:, n_num:]

    # Manually inverse only the scaler (skip imputer)
    scaler = num_pipe.named_steps['scaler']
    synth_num = scaler.inverse_transform(synth_num_scaled)

    # Categorical inverse
    ohe = cat_pipe.named_steps['onehot']
    synth_cat = ohe.inverse_transform(synth_cat_encoded)

    synth_array = np.column_stack([synth_num, synth_cat])
    
    original_columns = num_cols + cat_cols
    return pd.DataFrame(synth_array, columns=original_columns)


In [50]:
# generar 100 por clase
synth_0 = generate_copula_samples(copula_0, 1000, preprocessor)
synth_1 = generate_copula_samples(copula_1, 1000, preprocessor)

# convertir a dataframe y agregar el targer
df_synth_0 = pd.DataFrame(synth_0, columns=df_0.columns)
df_synth_0[target_col] = 0
df_synth_1 = pd.DataFrame(synth_1, columns=df_1.columns)
df_synth_1[target_col] = 1

# data augmentado
df_aug = pd.concat([df, df_synth_0, df_synth_1], ignore_index=True)
df_aug.head()

,duracion_s,rms_mean,rms_std,dementia,dementia type,gender,ethnicity,f0_mean,f0_std,f0_range,...,propn_ratio,noun_pron_ratio,syntax_depth,std_words_per_sent,mean_word_length,std_word_length,function_ratio,duration,pause_prob,wpm
0,71.019688,0.07139,0.050389,1,Alzheimer,male,Caucasian/White,174.444418,51.103025,311.05542,...,0.037975,0.465517,7.666667,75.330383,3.78903,2.132184,0.611814,72.76,0.263634,195.437053
1,47.0,0.144582,0.090372,1,Dementia,female,Black/African American,292.169978,115.242586,659.783615,...,0.0,0.5,5.5,13.379088,3.810811,1.850315,0.628378,47.04,0.122493,188.77551
2,38.0,0.12396,0.079574,1,Dementia,female,Black/African American,238.856939,58.060892,352.463261,...,0.0,0.655172,5.428571,12.290697,3.625,1.952562,0.585938,38.4,0.000328,200.0
3,90.0,0.061281,0.049997,1,Dementia,female,Black/African American,255.297147,68.547493,638.292396,...,0.0,1.146341,5.357143,9.598044,4.003597,2.092292,0.57554,89.0,0.002673,187.41573
4,55.0,0.06031,0.051383,1,Dementia,male,Caucasian/White,134.696479,38.961565,243.928633,...,0.067073,1.0,4.5,8.890444,4.286585,2.216252,0.640244,55.0,0.149834,178.909091


In [51]:
d_bayes = validate_augmented_data(df, df_aug)
d_bayes

Validación de datos médicos

Estadísticas por variable:
duracion_s: Original=56.02, Sintético=55.47, diff=1.0%
rms_mean: Original=0.09, Sintético=0.09, diff=0.8%
rms_std: Original=0.06, Sintético=0.06, diff=1.2%
dementia: Original=0.37, Sintético=0.48, diff=29.5%
f0_mean: Original=178.47, Sintético=175.78, diff=1.5%
f0_std: Original=48.84, Sintético=46.38, diff=5.1%
f0_range: Original=300.61, Sintético=295.10, diff=1.8%
articulation_rate: Original=1.85, Sintético=1.86, diff=0.7%
silence_segments_n: Original=54.00, Sintético=53.50, diff=0.9%
silence_dur_mean_s: Original=0.28, Sintético=0.28, diff=1.1%

Distribuciones categóricas:

dementia type
 Lewy body: original=5.0%, sintético=0.5%
 Parkinson: original=1.4%, sintético=0.1%
 Alzheimer: original=27.5%, sintético=2.7%
 Dementia: original=24.3%, sintético=2.4%
 no dementia: original=40.8%, sintético=94.2%
 Primary progressive aphasia: original=0.9%, sintético=0.1%

gender
 male: original=85.6%, sintético=53.9%
 female: original=14.4%, s

,type,variable,category,original,synthetic,diff_pct
0,numeric_mean,duracion_s,NaN,56.019615,55.466443,0.987460
1,numeric_mean,rms_mean,NaN,0.089943,0.090683,0.822961
2,numeric_mean,rms_std,NaN,0.061699,0.062422,1.170599
3,numeric_mean,dementia,NaN,0.371105,0.480663,29.522163
4,numeric_mean,f0_mean,NaN,178.469345,175.780553,1.506585
5,numeric_mean,f0_std,NaN,48.843398,46.375183,5.053324
6,numeric_mean,f0_range,NaN,300.610748,295.101545,1.832670
7,numeric_mean,articulation_rate,NaN,1.847707,1.859954,0.662846
8,numeric_mean,silence_segments_n,NaN,53.997167,53.496180,0.927802
9,numeric_mean,silence_dur_mean_s,NaN,0.281062,0.277861,1.139167


In [41]:
df_aug.shape

(2353, 74)

In [42]:
# guardar el dataset
df_aug.to_csv('data_aumentada_copulas.csv')

## Redes neuronales

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [8]:
# identifica columnas
num_cols = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] and col != 'dementia']
cat_cols = ['dementia type', 'gender', 'ethnicity']
target_col = 'dementia'

In [13]:
# preprocesing: imputacion y encoding

num_transformer = Pipeline(steps=[
  ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

X = preprocessor.fit_transform(df.drop(columns=[target_col]))
y = df[target_col].values


# Dataset para PyTorch

dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# dimesion de input
input_dim = X.shape[1]
latent_dim = 20
num_classes = 2

# modelo VAE condicionado en clase
class ConditionalVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, num_classes):
        super(ConditionalVAE, self).__init__()
        # encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + num_classes, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)
        # decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + num_classes, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def encode(self, x, c):
        h = self.encoder(torch.cat([x, c], dim=1))
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, c):
        return self.decoder(torch.cat([z, c], dim=1))

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, c), mu, logvar

# funcion de perdida VAE
def vae_loss(recon_x, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

# entrenamiento
model = ConditionalVAE(input_dim, latent_dim, num_classes)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y in dataloader:
        c = torch.nn.functional.one_hot(batch_y, num_classes=num_classes).float()
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(batch_x, c)
        loss = vae_loss(recon_batch, batch_x, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    print(f'Epoch {epoch+1}: Loss{train_loss/len(dataloader.dataset)}')

# generar datos augmentados
def generate_samples(model, num_samples, class_label):
    model.eval()
    with torch.no_grad():
        c = torch.nn.functional.one_hot(torch.full((num_samples,), class_label, dtype=torch.long), num_classes=num_classes).float()
        z = torch.randn(num_samples, latent_dim)
        samples = model.decode(z, c).numpy()
        return samples

# generar 100 por clase
synth_0 = generate_samples(model, 100, 0)
synth_1 = generate_samples(model, 100, 1)


Epoch 1: Loss73.2091282312323
Epoch 2: Loss71.94137089110636
Epoch 3: Loss71.1836689200685
Epoch 4: Loss70.50083003706027
Epoch 5: Loss68.78261735824282
Epoch 6: Loss64.13859244716741
Epoch 7: Loss61.44804299114109
Epoch 8: Loss60.24197004064284
Epoch 9: Loss58.495187732383144
Epoch 10: Loss56.84163224865965
Epoch 11: Loss54.94286852180114
Epoch 12: Loss53.22358193546111
Epoch 13: Loss51.66634267315311
Epoch 14: Loss50.256942630151855
Epoch 15: Loss49.5349654124749
Epoch 16: Loss48.725341461873256
Epoch 17: Loss48.330713169432904
Epoch 18: Loss47.248199484503644
Epoch 19: Loss46.724582931475325
Epoch 20: Loss46.40869064979445
Epoch 21: Loss45.852225338771056
Epoch 22: Loss44.61967308325403
Epoch 23: Loss44.631407448657846
Epoch 24: Loss43.964752164846104
Epoch 25: Loss43.63428000215093
Epoch 26: Loss43.532262710268704
Epoch 27: Loss42.947394849220686
Epoch 28: Loss42.633895528215184
Epoch 29: Loss42.38079064561017
Epoch 30: Loss42.28830161378336
Epoch 31: Loss42.165855537392936
Epoch 3

In [15]:
#df_synth = pd.concat([synth_0, synth_1], ignore_index=True)
#df_augmentado = pd.concat([df, df_synth], ignore_index=True)    